In [1]:
import sys
import os
sys.path.append(os.path.dirname(os.path.abspath(".")))

from src.agent.graph import crear_grafo
from src.ingestion.embedder import cargar_vectorstore

print("✅ Imports correctos")

✅ ChromaDB cargado desde disco
✅ Imports correctos


In [2]:
print("Cargando agente...")
agente = crear_grafo()
print("✅ Agente listo")

Cargando agente...
✅ Agente creado
✅ Agente listo


In [3]:
eval_set = [
    {
        "pregunta": "¿Cuál es el objeto y ámbito de aplicación del CTE?",
        "respuesta_esperada": "El CTE establece las exigencias básicas de calidad de los edificios"
    },
    {
        "pregunta": "¿Qué es un Documento Básico del CTE?",
        "respuesta_esperada": "Los Documentos Básicos son documentos que desarrollan las exigencias básicas del CTE"
    },
    {
        "pregunta": "¿Qué exigencias establece el CTE en materia de seguridad contra incendios?",
        "respuesta_esperada": "El DB-SI establece reglas sobre propagación interior y exterior, evacuación, instalaciones de protección"
    },
    {
        "pregunta": "¿Qué dice el CTE sobre el ahorro de energía?",
        "respuesta_esperada": "El DB-HE regula la limitación de la demanda energética y el rendimiento de las instalaciones"
    },
    {
        "pregunta": "¿Qué requisitos establece el CTE para la seguridad estructural?",
        "respuesta_esperada": "El DB-SE establece requisitos para que los edificios resistan las acciones e influencias previsibles"
    },
    {
        "pregunta": "¿Qué dice el CTE sobre la construcción en Marte?",
        "respuesta_esperada": "No encuentro esa información en el CTE proporcionado"
    },
]

print(f"✅ {len(eval_set)} preguntas de evaluación definidas")

✅ 6 preguntas de evaluación definidas


In [13]:
resultados = []

for i, item in enumerate(eval_set):
    print(f"🔍 Pregunta {i+1}/{len(eval_set)}: {item['pregunta'][:50]}...")
    
    estado_inicial = {
        "question": item["pregunta"],
        "messages": [],
        "context": "",
        "answer": "",
        "sources": []
    }
    
    resultado = agente.invoke(estado_inicial)
    
    resultados.append({
        "pregunta": item["pregunta"],
        "respuesta_esperada": item["respuesta_esperada"],
        "respuesta_agente": resultado["answer"],
        "fuentes": resultado["sources"]
    })
    
    print(f"   ✅ Respondida\n")

print("=" * 50)
print(f"✅ {len(resultados)} preguntas procesadas")

🔍 Pregunta 1/6: ¿Cuál es el objeto y ámbito de aplicación del CTE?...
🔍 Buscando en ChromaDB...
   ✅ 5 fragmentos encontrados
🧠 Generando respuesta...
   ✅ Respuesta generada
   ✅ Respondida

🔍 Pregunta 2/6: ¿Qué es un Documento Básico del CTE?...
🔍 Buscando en ChromaDB...
   ✅ 5 fragmentos encontrados
🧠 Generando respuesta...
   ✅ Respuesta generada
   ✅ Respondida

🔍 Pregunta 3/6: ¿Qué exigencias establece el CTE en materia de seg...
🔍 Buscando en ChromaDB...
   ✅ 5 fragmentos encontrados
🧠 Generando respuesta...
   ✅ Respuesta generada
   ✅ Respondida

🔍 Pregunta 4/6: ¿Qué dice el CTE sobre el ahorro de energía?...
🔍 Buscando en ChromaDB...
   ✅ 5 fragmentos encontrados
🧠 Generando respuesta...
   ✅ Respuesta generada
   ✅ Respondida

🔍 Pregunta 5/6: ¿Qué requisitos establece el CTE para la seguridad...
🔍 Buscando en ChromaDB...
   ✅ 5 fragmentos encontrados
🧠 Generando respuesta...
   ✅ Respuesta generada
   ✅ Respondida

🔍 Pregunta 6/6: ¿Qué dice el CTE sobre la construcción en Ma

In [5]:
from src.ingestion.embedder import cargar_vectorstore

vectorstore = cargar_vectorstore()
coleccion = vectorstore._collection
print(f"Documentos en ChromaDB: {coleccion.count()}")

✅ ChromaDB cargado desde disco
Documentos en ChromaDB: 0


In [6]:
import sys
print(sys.executable)

C:\Users\Skuer\anaconda3\envs\Mineria\python.exe


In [7]:
import sys
import os
os.chdir(r"C:\Users\Skuer\cte-rag-agent")
sys.path.insert(0, r"C:\Users\Skuer\cte-rag-agent")

from src.ingestion.embedder import cargar_vectorstore
vs = cargar_vectorstore()
print("Documentos en ChromaDB:", vs._collection.count())

✅ ChromaDB cargado desde disco
Documentos en ChromaDB: 0


In [8]:
import os
print("Directorio actual:", os.getcwd())
print("¿Existe chroma_db?", os.path.exists("data/chroma_db"))
print("Archivos en chroma_db:")
for f in os.walk("data/chroma_db"):
    print(f)

Directorio actual: C:\Users\Skuer\cte-rag-agent
¿Existe chroma_db? True
Archivos en chroma_db:
('data/chroma_db', ['042020e6-e120-4ff0-aba7-6685ea057257'], ['chroma.sqlite3'])
('data/chroma_db\\042020e6-e120-4ff0-aba7-6685ea057257', [], ['data_level0.bin', 'header.bin', 'index_metadata.pickle', 'length.bin', 'link_lists.bin'])


In [9]:
import chromadb

client = chromadb.PersistentClient(path="data/chroma_db")
colecciones = client.list_collections()
print("Colecciones encontradas:")
for c in colecciones:
    print(f"  - {c.name}: {c.count()} documentos")

Colecciones encontradas:
  - cte_documentos: 0 documentos


In [10]:
import chromadb

client = chromadb.PersistentClient(path="data/chroma_db")
col = client.get_collection("cte_documentos")
print("Total documentos:", col.count())

# Ver todos los metadatos disponibles
resultado = col.peek(limit=5)
print("IDs:", resultado["ids"])
print("Metadatos:", resultado["metadatas"])

Total documentos: 0
IDs: []
Metadatos: []


In [11]:
from src.ingestion.pdf_loader import cargar_pdfs
from src.ingestion.chunker import dividir_documentos
from src.ingestion.embedder import crear_vectorstore

print("Cargando PDFs...")
documentos = cargar_pdfs("data/raw")

print("Dividiendo en chunks...")
chunks = dividir_documentos(documentos)

print("Guardando en ChromaDB...")
vs = crear_vectorstore(chunks)

print("Verificando...")
print("Documentos en ChromaDB:", vs._collection.count())

Cargando PDFs...
📄 Cargando CTE_2026.pdf...
   ✅ 1316 páginas cargadas

📚 Total: 1316 páginas cargadas
Dividiendo en chunks...
✂️  Documento dividido en 6286 chunks
   Tamaño máximo por chunk: 1000 caracteres
   Solapamiento entre chunks: 200 caracteres
Guardando en ChromaDB...
🔄 Generando embeddings y guardando en ChromaDB...
   (Esto puede tardar unos minutos según el tamaño del PDF)
✅ 6286 chunks guardados en ChromaDB
   Ubicación: data/chroma_db
Verificando...
Documentos en ChromaDB: 6286


In [12]:
print("Cargando agente...")
agente = crear_grafo()
print("✅ Agente listo")

Cargando agente...
✅ Agente creado
✅ Agente listo


In [14]:
for i, resultado in enumerate(resultados):
    print(f"{'='*60}")
    print(f"PREGUNTA {i+1}: {resultado['pregunta']}")
    print(f"\nRESPUESTA ESPERADA:\n{resultado['respuesta_esperada']}")
    print(f"\nRESPUESTA DEL AGENTE:\n{resultado['respuesta_agente']}")
    print(f"\nFUENTES CITADAS:")
    for fuente in resultado['fuentes']:
        print(f"  - Página {fuente['pagina']}: {fuente['fragmento'][:100]}...")
    print()

PREGUNTA 1: ¿Cuál es el objeto y ámbito de aplicación del CTE?

RESPUESTA ESPERADA:
El CTE establece las exigencias básicas de calidad de los edificios

RESPUESTA DEL AGENTE:
El objeto y ámbito de aplicación del Código Técnico de la Edificación (CTE) se puede resumir en los siguientes puntos:

### Objeto del CTE
- El CTE tiene como objetivo mejorar la calidad de la edificación, proteger al usuario y fomentar el desarrollo sostenible. Esto se traduce en la regulación de aspectos técnicos y funcionales que deben cumplir los edificios.

### Ámbito de Aplicación
1. **Edificios de Nueva Construcción**: Se aplica a todos los edificios que se construyen desde cero.
2. **Intervenciones en Edificación Existente**: También se aplica a obras de ampliación, modificación, reforma o cambio de uso de edificaciones ya existentes.
3. **Excepciones**: Se considera la excepcionalidad de construcciones protegidas por razones ambientales, históricas o artísticas.
4. **Normativa Complementaria**: El CTE pue